In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # Automatically loads .env if it exists
DATA_DIR = Path(os.getenv("DATA_DIR", "data"))


import pandas as pd
import numpy as np
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import shutil
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv(DATA_DIR / "MOCK_DATA.csv")
df['Delivery Date'] = pd.to_datetime(df['Delivery Date'])
 
print("Total orders:", len(df))
print("Date range:", df['Delivery Date'].min().date(), "→", df['Delivery Date'].max().date())
print("\nOrders per product category:")
print(df['Item'].value_counts())

In [ ]:
df['week'] = df['Delivery Date'].dt.to_period('W').apply(lambda r: r.start_time)
 
df_weekly = df.groupby(['week', 'Item']).size().reset_index(name='target')
 
# Rename for clarity
df_weekly = df_weekly.rename(columns={'week': 'timestamp', 'Item': 'item_id'})
 
print("\nWeekly aggregated data shape:", df_weekly.shape)
print(df_weekly.head(10))

In [ ]:
ts_df = TimeSeriesDataFrame.from_data_frame(
    df_weekly,
    id_column="item_id",
    timestamp_column="timestamp"
)
 
# Fill any missing weeks with 0 orders (closed weeks, holidays, etc.)
ts_df = ts_df.convert_frequency(freq="W")
ts_df = ts_df.fill_missing_values(method="constant", value=0.0)
 
print(f"\nProduct series: {ts_df.index.get_level_values(0).nunique()}")
print(f"Date range: {ts_df.index.get_level_values(1).min().date()} → "
      f"{ts_df.index.get_level_values(1).max().date()}")
print(ts_df.head(10))

In [ ]:
all_timestamps = ts_df.index.get_level_values(1)
test_end   = all_timestamps.max()
train_end  = test_end - pd.Timedelta(weeks=4)
 
train_df = ts_df.slice_by_time(
    start_time=all_timestamps.min(),
    end_time=train_end
)
 
print(f"\nTraining data ends:  {train_end.date()}")
print(f"Test period ends:    {test_end.date()}")
print(f"Training weeks per product: ~{len(train_df) // ts_df.index.get_level_values(0).nunique()}")

In [ ]:
predictor = TimeSeriesPredictor(
    prediction_length=1,              # 1 week ahead (= 7 days, in weekly freq)
    path="chronos_bolt_flower_model",
    eval_metric="RMSE",
    freq="W",
).fit(
    train_df,
    hyperparameters={
        "Chronos": [
            {
                "model_path": "bolt_tiny",       # CPU-compatible, fast
                "fine_tune": True,
                "fine_tune_steps": 300,          # conservative — small dataset
                "fine_tune_lr": 1e-5,
                "fine_tune_batch_size": 8,       # small batch for small data
                "eval_during_fine_tune": True,   # picks best checkpoint
            }
        ]
    },
    enable_ensemble=False,
    time_limit=1800,   # 30 minutes max (should finish much faster)
)

In [ ]:
score = predictor.evaluate(ts_df)

rmse = score.get("RMSE")
print(f"Test RMSE: {rmse:.4f}")
# Lower RMSE = model is closer to actual order counts
# On weekly data, an RMSE of 1–2 means predictions are off by 1–2 orders/week
# which is acceptable for inventory planning

In [ ]:
next_week_forecast = predictor.predict(ts_df)
forecast_df = next_week_forecast.reset_index()
forecast_df = forecast_df.rename(columns={
    'item_id': 'Product',
    'timestamp': 'Week Starting',
    'mean': 'Predicted Orders',
    '0.1': 'Low Estimate',
    '0.9': 'High Estimate',
})
 
# Round to whole orders
forecast_df['Predicted Orders'] = forecast_df['Predicted Orders'].round().astype(int)
 
print("\n" + "="*60)
print("NEXT WEEK FORECAST — Orders per Product")
print("="*60)
print(forecast_df[['Product', 'Week Starting', 'Predicted Orders']].to_string(index=False))

In [ ]:
print("\n" + "="*60)
print("FLOWER MIX BY PRODUCT (Historical Percentages)")
print("Use with the forecast above to calculate stock needed.")
print("="*60)
 
original_df = pd.read_csv(DATA_DIR / "MOCK_DATA.csv")
for item in sorted(original_df['Item'].unique()):
    item_total = len(original_df[original_df['Item'] == item])
    top_flowers = (
        original_df[original_df['Item'] == item]['Flower Choice']
        .value_counts()
        .head(8)
    )
    print(f"\n  {item} (total {item_total} orders):")
    for flower, count in top_flowers.items():
        pct = count / item_total * 100
        print(f"    {flower:<35} {count:>3} orders ({pct:.1f}%)")

In [ ]:
print("\n" + "="*60)
print("HISTORICAL MONTHLY ORDERS PER PRODUCT")
print("Use this to spot seasonal peaks in YOUR real data.")
print("="*60)
 
original_df['Delivery Date'] = pd.to_datetime(original_df['Delivery Date'])
original_df['Month'] = original_df['Delivery Date'].dt.strftime('%Y-%m')
monthly = original_df.groupby(['Month', 'Item']).size().unstack(fill_value=0)
print(monthly.to_string())

In [ ]:
shutil.make_archive("chronos_bolt_flower_model", "zip", "chronos_bolt_flower_model")
print("\n✅ Done! Download: chronos_bolt_flower_model.zip")
print("   (Right-click the zip in the left sidebar → Download)")